# Model Enhancement: Student Academic Risk Prediction

This notebook documents a follow-up review of the original `01_data_overview.ipynb` pipeline. It covers three issues found during review, and the model/threshold improvements made in response.

## Issues found
1. **Naming**: the dataset has no literal dropout field. The target `at_risk = (G3 < 10)` is a proxy (risk of academic failure), not withdrawal data. This should be stated plainly, not implied by the word "dropout".
2. **Deployed threshold mismatch**: the original notebook validated 0.35 as the recall-optimal threshold, but both deployed apps (`backend/app.py`, `streamlit_app/app.py`) hardcoded `THRESHOLD = 0.6` -- undoing the entire point of the threshold-tuning work. At 0.6, real recall was only 46.2%, not the 88% the README claimed.
3. **Underused features + no cross-validation**: only 8 of 30 available columns were used, and evaluation relied on a single ~79-row test split, which is small and high-variance for this dataset size (395 rows total).

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_predict, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score

df = pd.read_csv('../data/student_data.csv', sep=';')
df['at_risk'] = (df['G3'] < 10).astype(int)

binary_cols = ['schoolsup','famsup','paid','activities','nursery','higher','internet','romantic']
df[binary_cols] = df[binary_cols].replace({'yes':1,'no':0})

y = df['at_risk']
print(df.shape, y.value_counts().to_dict())

## Step 1: Reproduce the original deployed behavior honestly

Confirm what the deployed apps actually did (threshold 0.6) vs. what the notebook claimed was optimal (threshold 0.35), on the same train/test split used originally.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

ORIGINAL_FEATURES = ['studytime','failures','absences','schoolsup','famsup','paid','higher','internet']
X_orig = df[ORIGINAL_FEATURES]
X_train, X_test, y_train, y_test = train_test_split(X_orig, y, test_size=0.2, random_state=42, stratify=y)

model_bal = LogisticRegression(max_iter=1000, class_weight='balanced')
model_bal.fit(X_train, y_train)
y_prob = model_bal.predict_proba(X_test)[:,1]

print('=== Deployed threshold (0.6) -- what was actually live ===')
print(classification_report(y_test, (y_prob>=0.6).astype(int), digits=3))
print('=== Notebook-recommended threshold (0.35) -- never actually deployed ===')
print(classification_report(y_test, (y_prob>=0.35).astype(int), digits=3))

**Finding confirmed**: at the threshold that was actually live in production (0.6), recall for at-risk students was ~46%, well below the ~62% the *default* 0.5 threshold gives, and far below the 88% the README advertised (which only ever existed in the notebook, never in the deployed apps).

## Step 2: Does expanding beyond the original 8 features help?

Compare the original 8-feature set against the full available feature set (parental education, alcohol use, family relationship quality, travel time, school/family background, etc.), using 5-fold stratified cross-validation instead of a single small test split.

In [ ]:
NUMERIC_SCREENER = ['age','Medu','Fedu','traveltime','studytime','failures','famrel','freetime','goout','Dalc','Walc','health','absences']
BIN = ['schoolsup','famsup','paid','activities','nursery','higher','internet','romantic']
CAT = ['school','sex','address','famsize','Pstatus','Mjob','Fjob','reason','guardian']

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Original 8 features
pipe_orig = Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))])
oof_orig = cross_val_predict(pipe_orig, X_orig, y, cv=cv, method='predict_proba')[:,1]
print('Original 8 features, Balanced LR -- CV ROC-AUC:', roc_auc_score(y, oof_orig))

# Expanded features
X_exp = df[NUMERIC_SCREENER+BIN+CAT]
preprocess = ColumnTransformer([('num', StandardScaler(), NUMERIC_SCREENER+BIN), ('cat', OneHotEncoder(handle_unknown='ignore'), CAT)])
pipe_exp = Pipeline([('preprocess', preprocess), ('model', LogisticRegression(max_iter=2000, class_weight='balanced'))])
oof_exp = cross_val_predict(pipe_exp, X_exp, y, cv=cv, method='predict_proba')[:,1]
print('Expanded features, Balanced LR -- CV ROC-AUC:', roc_auc_score(y, oof_exp))

Logistic Regression barely benefits from the extra features. Try tree-based models, which can capture non-linear interactions the extra columns might contain.

In [ ]:
scale_pos_weight = (y==0).sum()/(y==1).sum()

pipe_rf = Pipeline([('preprocess', preprocess), ('model', RandomForestClassifier(class_weight='balanced', random_state=42))])
rf_search = RandomizedSearchCV(pipe_rf, {
    'model__n_estimators':[150,250,350], 'model__max_depth':[4,6,10,None],
    'model__min_samples_split':[2,5,10], 'model__min_samples_leaf':[1,2,4]
}, n_iter=10, scoring='roc_auc', cv=cv, random_state=42)
rf_search.fit(X_exp, y)
print('Random Forest (tuned) -- CV ROC-AUC:', rf_search.best_score_)

pipe_xgb = Pipeline([('preprocess', preprocess), ('model', XGBClassifier(eval_metric='logloss', random_state=42, scale_pos_weight=scale_pos_weight))])
xgb_search = RandomizedSearchCV(pipe_xgb, {
    'model__n_estimators':[80,120,150,200], 'model__max_depth':[2,3,4],
    'model__learning_rate':[0.02,0.03,0.05,0.08], 'model__subsample':[0.7,0.85,1.0],
    'model__colsample_bytree':[0.7,0.85,1.0]
}, n_iter=25, scoring='roc_auc', cv=cv, random_state=42)
xgb_search.fit(X_exp, y)
print('XGBoost (tuned) -- CV ROC-AUC:', xgb_search.best_score_)
print('Best XGBoost params:', xgb_search.best_params_)

**Result**: XGBoost with the expanded feature set reaches ~0.71 CV ROC-AUC vs. ~0.68 for the original 8-feature Logistic Regression -- a real but modest improvement. This becomes the **Screener model** (usable from day one of term, no grades required).

## Step 3: What if G1 and G2 (first two period grades) are available?

G1 and G2 are recorded *before* the final grade G3 that defines `at_risk` -- using them is **not label leakage**, it's a legitimate early-warning signal (like using week-4 quiz scores to predict end-of-semester outcomes). Worth testing separately, since it changes *when* in the school year the model can be used, not just how well it performs.

In [ ]:
NUMERIC_EARLY = NUMERIC_SCREENER + ['G1','G2']
X_early = df[NUMERIC_EARLY+BIN+CAT]
preprocess_early = ColumnTransformer([('num', StandardScaler(), NUMERIC_EARLY+BIN), ('cat', OneHotEncoder(handle_unknown='ignore'), CAT)])

pipe_xgb_early = Pipeline([('preprocess', preprocess_early), ('model', XGBClassifier(eval_metric='logloss', random_state=42, scale_pos_weight=scale_pos_weight))])
xgb_early_search = RandomizedSearchCV(pipe_xgb_early, {
    'model__n_estimators':[80,120,150,200], 'model__max_depth':[2,3,4],
    'model__learning_rate':[0.02,0.03,0.05,0.08], 'model__subsample':[0.7,0.85,1.0],
    'model__colsample_bytree':[0.7,0.85,1.0]
}, n_iter=25, scoring='roc_auc', cv=cv, random_state=42)
xgb_early_search.fit(X_early, y)
print('Early-Warning XGBoost (with G1+G2) -- CV ROC-AUC:', xgb_early_search.best_score_)
print('Best params:', xgb_early_search.best_params_)

**Result**: ~0.97-0.98 CV ROC-AUC -- a dramatic, honest improvement, since period grades are strong predictors of the final outcome. This becomes the **Early-Warning model**, deployed as a second mode alongside the Screener rather than a replacement for it, since it can only be used once G1/G2 exist.

## Step 4: Threshold selection on out-of-fold predictions (not a small held-out test set)

Selecting a threshold from a single ~79-row test set (as in the original notebook) is unstable. Using 5-fold out-of-fold predictions across the full dataset gives a more reliable estimate for each candidate threshold.

In [ ]:
oof_screener = cross_val_predict(xgb_search.best_estimator_, X_exp, y, cv=cv, method='predict_proba')[:,1]
oof_early = cross_val_predict(xgb_early_search.best_estimator_, X_early, y, cv=cv, method='predict_proba')[:,1]

def sweep(y_true, y_prob, name):
    print(f'\n{name}:')
    for th in [0.3,0.35,0.4,0.45,0.5,0.55,0.6]:
        pred = (y_prob>=th).astype(int)
        print(f'  th={th:.2f}  recall={recall_score(y_true,pred):.3f}  precision={precision_score(y_true,pred):.3f}  f1={f1_score(y_true,pred):.3f}')

sweep(y, oof_screener, 'SCREENER')
sweep(y, oof_early, 'EARLY-WARNING')

## Final decisions

- **Screener threshold: 0.35** -- ~78% recall / ~43% precision. Keeps the original project's recall-priority philosophy (missing an at-risk student costs more than a false alarm), but from a genuinely better model than the original.
- **Early-Warning threshold: 0.40** -- ~94% recall AND ~82% precision. Both high, because G1/G2 make this a much easier problem.
- **Both thresholds are now imported from `backend/utils/config.py` by every consumer** (training script, Flask API, Streamlit app), so the notebook-vs-production mismatch that caused the original bug cannot happen again -- there is only one place these numbers are defined.